In [7]:
import sys
import os
import pandas as pd
import py4cytoscape as p4c
import IPython

print("Working directory:", os.getcwd())

Working directory: /home/jovyan/work/persistent/AOP-RareDiseases-KG/ppi


In [8]:
print(f"Loading Javascript client ... {p4c.get_browser_client_channel()} on {p4c.get_jupyter_bridge_url()}")
browser_client_js = p4c.get_browser_client_js()
IPython.display.Javascript(browser_client_js)

Loading Javascript client ... 48c13e29-5a4c-4c97-80b0-162e3d7ef9bb on https://jupyter-bridge.cytoscape.org


<IPython.core.display.Javascript object>

In [11]:
import py4cytoscape as p4c

print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!


'You are connected to Cytoscape!'

In [12]:
import os
import logging

# ── Fix log directory ─────────────────────────────────────────────────────────
os.makedirs("/home/jovyan/work/persistent/AOP-RareDiseases-KG/ppi/logs", exist_ok=True)
logging.getLogger("py4cytoscape").handlers = []
logging.getLogger("py4cytoscape").addHandler(logging.NullHandler())

import pandas as pd
import py4cytoscape as p4c

pd.set_option('display.max_colwidth', None)
print("Working directory:", os.getcwd())

# ── Load PPI data ─────────────────────────────────────────────────────────────
df = pd.read_csv("intact_human_ppi_annotated.tsv", sep="\t", dtype=str)
df["mi_score"] = pd.to_numeric(df["mi_score"], errors="coerce")

# Optional: filter to high-confidence interactions
df = df.dropna(subset=["geneA_symbol", "geneB_symbol", "mi_score"])
df = df[df["mi_score"] >= 0.45]
print(f"Loaded: {len(df):,} interactions | "
      f"{pd.concat([df['geneA_symbol'], df['geneB_symbol']]).nunique():,} unique proteins")

# ── Build node table ──────────────────────────────────────────────────────────
genes_A = df[["uniprotA", "geneA_symbol", "geneA_entrez", "geneA_ensembl"]].rename(
    columns={"uniprotA": "uniprot", "geneA_symbol": "id",
             "geneA_entrez": "entrez_id", "geneA_ensembl": "ensembl_id"}
)
genes_B = df[["uniprotB", "geneB_symbol", "geneB_entrez", "geneB_ensembl"]].rename(
    columns={"uniprotB": "uniprot", "geneB_symbol": "id",
             "geneB_entrez": "entrez_id", "geneB_ensembl": "ensembl_id"}
)

all_nodes = (
    pd.concat([genes_A, genes_B], ignore_index=True)
    .drop_duplicates(subset="id")
    .assign(label=lambda x: x["id"], group="protein", type="gene")   # ← add type="gene"
)
print(f"Nodes: {len(all_nodes):,}")

# ── Build edge table ──────────────────────────────────────────────────────────
all_edges = (
    df[["geneA_symbol", "geneB_symbol", "mi_score"]]
    .drop_duplicates()
    .rename(columns={"geneA_symbol": "source", "geneB_symbol": "target"})
    .assign(interaction="interacts_with")
)
print(f"Edges: {len(all_edges):,}")

# ── Connect to Cytoscape ──────────────────────────────────────────────────────
p4c.cytoscape_ping()
print("Connected to Cytoscape")

# ── Create network ────────────────────────────────────────────────────────────
suid = p4c.create_network_from_data_frames(
    nodes=all_nodes,
    edges=all_edges,
    source_id_list="source",
    target_id_list="target",
    interaction_type_list="interaction",
    node_id_list="id",
    title="IntAct Human PPI Network",
    collection="AOP-RareDiseases-KG",
)
print(f"Network created — SUID: {suid}")

# ── Load node attributes ──────────────────────────────────────────────────────
p4c.load_table_data(
    data=all_nodes[["id", "label", "uniprot", "entrez_id", "ensembl_id", "group", "type"]],  # ← add "type"
    data_key_column="id",
    table="node",
    table_key_column="name",
    network=suid,
)
print("Node attributes loaded")

# ── Load edge attributes (mi_score as weight) ─────────────────────────────────
p4c.load_table_data(
    data=all_edges[["source", "target", "mi_score"]],
    data_key_column="source",
    table="edge",
    table_key_column="source",
    network=suid,
)


# ── Save ──────────────────────────────────────────────────────────────────────
p4c.save_session("intact_ppi_network.cys")
print("Saved → intact_ppi_network.cys")
print(f"\nSummary — Nodes: {len(all_nodes):,} | Edges: {len(all_edges):,}")

Working directory: /home/jovyan/work/persistent/AOP-RareDiseases-KG/ppi
Loaded: 19,894 interactions | 1,194 unique proteins
Nodes: 1,194
Edges: 2,157
You are connected to Cytoscape!
Connected to Cytoscape
Applying default style...
Applying preferred layout
Network created — SUID: 707342
Node attributes loaded
This file has been overwritten.
Saved → intact_ppi_network.cys

Summary — Nodes: 1,194 | Edges: 2,157
